<a href="https://colab.research.google.com/github/dgaida/rag_foerderkatalog/blob/master/notebooks/RAG_Foerderkatallog_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 RAG Förderkatalog v0.2.0 - Google Colab

[![GitHub Release](https://img.shields.io/badge/release-v0.3.0-blue.svg)](https://github.com/dgaida/rag_foerderkatalog/releases/tag/v0.3.0)
[![Python 3.11+](https://img.shields.io/badge/python-3.11+-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**Semantische Suche in deutschen Forschungsförderprojekten**

Dieses Notebook ermöglicht die einfache Nutzung der RAG Förderkatalog-Anwendung in Google Colab:
- ✅ **HuggingFace Embeddings** statt Ollama (Cloud-kompatibel)
- ✅ **Vorbereiteter Index** (~300k Projekte)
- ✅ **Keine lokale Installation** nötig
- ✅ **Gradio Web-UI** im Browser

---

## 📋 Voraussetzungen

- **GROQ API Key** für LLM-Funktionen (kostenlos unter [console.groq.com](https://console.groq.com/))
- **Google Drive** wird temporär für Downloads genutzt (~2GB)
- **Runtime**: Standard-Python (kein GPU nötig)

---

## 🚀 Schritt 1: Installation der Dependencies

In [1]:
%%capture
# Basis-Pakete installieren (dauert ~2-3 Minuten)
!pip install --upgrade pip setuptools wheel

# Core Dependencies
!pip install pandas numpy faiss-cpu gradio python-dotenv tqdm requests

# LLM Client
!pip install git+https://github.com/dgaida/llm_client.git

# HuggingFace Embeddings Support
!pip install llama-index-embeddings-huggingface

print("✅ Dependencies installiert!")

In [2]:
%%capture
# RAG Förderkatalog v0.3.1 installieren
!pip install git+https://github.com/dgaida/rag_foerderkatalog.git@v0.3.1

print("✅ RAG Förderkatalog v0.3.1 installiert!")

## 📥 Schritt 2: Download des vorbereiteten Index

In [ ]:
import os
import zipfile
from pathlib import Path
import requests
from tqdm import tqdm

# Stelle sicher, dass wir in /content sind
os.chdir('/content')

# Download URL für v0.3.0
RELEASE_VERSION = "v0.3.1"
RELEASE_URL = f"https://github.com/dgaida/rag_foerderkatalog/releases/download/{RELEASE_VERSION}/rag_foerderkatalog_index_{RELEASE_VERSION}.zip"
ZIP_FILE = "/content/rag_complete.zip"

print("═" * 60)
print("  RAG Förderkatalog v0.3.1 - Complete Release")
print("═" * 60)
print("")
print("📥 Lade Complete Release...")
print(f"   Version: {RELEASE_VERSION}")
print(f"   URL: {RELEASE_URL}")
print(f"   Größe: ~1 GB")
print("")
print("⏳ Dies kann 5-10 Minuten dauern...")
print("")

# Download mit Fortschrittsanzeige
try:
    response = requests.get(RELEASE_URL, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))

    with open(ZIP_FILE, 'wb') as f, tqdm(
        desc="📦 Download",
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))

    print("")
    print("✅ Download abgeschlossen")

except Exception as e:
    print(f"❌ Download-Fehler: {e}")
    print("")
    print("Mögliche Lösungen:")
    print("   1. Prüfen Sie Ihre Internetverbindung")
    print("   2. Stellen Sie sicher, dass das Release existiert:")
    print(f"      {RELEASE_URL}")
    print("   3. Versuchen Sie es später erneut")
    raise

print("")
print("📦 Entpacke Release...")
print(f"   Zielverzeichnis: /content/")

# Entpacken
try:
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"   Dateien im Archiv: {len(file_list)}")

        # Entpacke mit Progress Bar
        for file in tqdm(file_list, desc="📂 Entpacken"):
            zip_ref.extract(file, "/content/")

    print("✅ Entpacken abgeschlossen")

except Exception as e:
    print(f"❌ Entpack-Fehler: {e}")
    raise

print("")

# Aufräumen
if Path(ZIP_FILE).exists():
    os.remove(ZIP_FILE)
    print("🧹 ZIP-Datei gelöscht")

print("")
print("📊 Prüfe entpackte Dateien...")

# Erwartete Struktur nach Entpacken
RELEASE_DIR = Path(f"/content/rag_foerderkatalog_index_{RELEASE_VERSION}")
EXPECTED_FILES = {
    'input/foerderkatalog_export.csv': 'CSV',
    'data/vector_hf.index': 'Index',
    'data/embeddings_map_hf.json': 'Mapping'
}

all_ok = True
for rel_path, description in EXPECTED_FILES.items():
    file_path = RELEASE_DIR / rel_path
    exists = file_path.exists()
    status = "✅" if exists else "❌"

    print(f"   {status} {description}: {rel_path}")

    if exists:
        size = file_path.stat().st_size
        if size > 1e9:
            print(f"       Größe: {size / 1e9:.2f} GB")
        else:
            print(f"       Größe: {size / 1e6:.2f} MB")
    else:
        all_ok = False

print("")

if all_ok:
    print("✅ Alle erforderlichen Dateien gefunden!")
    print("")
    print(f"📁 Release-Verzeichnis: {RELEASE_DIR}")
    print("")
    print("Nächster Schritt: API Key konfigurieren")
else:
    print("❌ FEHLER: Nicht alle Dateien wurden korrekt entpackt!")
    print("")
    print("Bitte führen Sie diese Zelle erneut aus.")
    raise FileNotFoundError("Unvollständiger Release-Download")

## 🚀 Schritt 3: Anwendung starten

Die Gradio-Oberfläche wird automatisch geöffnet. Klicken Sie auf den generierten Link.

In [ ]:
import sys
from src.search.engine import ProjectSearchEngine
from src.app import build_ui
from src.utils.logging_config import setup_logging
import logging

print("═" * 60)
print("  RAG Förderkatalog v0.3.0 - Startup")
print("═" * 60)
print("")

# Stelle sicher dass wir in /content sind
os.chdir('/content')

# Pfade für Colab
COLAB_INPUT_DIR = RELEASE_DIR / "input"
COLAB_DATA_DIR = RELEASE_DIR / "data"
COLAB_CSV_FILE = COLAB_INPUT_DIR / "foerderkatalog_export.csv"

print("🔍 Prüfe Dateisystem...")
print(f"   Working Directory: {os.getcwd()}")
print(f"   Release Dir: {RELEASE_DIR.exists()}")
print(f"   Input Dir: {COLAB_INPUT_DIR.exists()}")
print(f"   Data Dir: {COLAB_DATA_DIR.exists()}")
print(f"   CSV: {COLAB_CSV_FILE.exists()}")
print("")

# Setup Logging
setup_logging(level=logging.INFO)

print("🔧 Konfiguration:")
print("   • Provider: HuggingFace")
print("   • Modell: intfloat/e5-small-v2")
print("   • CSV: Included in Release")
print("   • Index: Pre-loaded")
print("")

# ===== Config-Pfade für Colab setzen =====
import src.config as config

config.ROOT = RELEASE_DIR
config.INPUT_CSV = COLAB_CSV_FILE
config.DATA_DIR = COLAB_DATA_DIR
config.FAISS_INDEX_FILE_HF = COLAB_DATA_DIR / 'vector_hf.index'
config.EMBED_MAP_FILE_HF = COLAB_DATA_DIR / 'embeddings_map_hf.json'
config.PROGRESS_FILE_HF = COLAB_DATA_DIR / 'indexing_progress_hf.json'
config.LOG_DIR = Path('/content/logs')
config.LOG_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Config-Pfade für Colab gesetzt")
print(f"   Root: {config.ROOT}")
print(f"   CSV: {config.INPUT_CSV}")
print(f"   Data: {config.DATA_DIR}")
print(f"   Logs: {config.LOG_DIR}")
print("")

# ===== Validierung =====
print("📊 Validiere Dateien...")

required_files = {
    'CSV': config.INPUT_CSV,
    'Index': config.FAISS_INDEX_FILE_HF,
    'Mapping': config.EMBED_MAP_FILE_HF
}

missing_files = []
for name, file_path in required_files.items():
    if file_path.exists():
        size = file_path.stat().st_size
        size_str = f"{size/1e9:.2f} GB" if size > 1e9 else f"{size/1e6:.2f} MB"
        print(f"   ✅ {name}: {file_path.name} ({size_str})")
    else:
        print(f"   ❌ {name}: {file_path} (fehlt)")
        missing_files.append(name)

print("")

if missing_files:
    print("❌ FEHLER: Folgende Dateien fehlen:")
    for name in missing_files:
        print(f"   • {name}")
    print("")
    print("Bitte führen Sie Cell 2 (Download) erneut aus")
    sys.exit(1)

# ===== Engine initialisieren =====
print("⏳ Initialisiere Engine...")

try:
    # Engine mit HuggingFace Provider
    engine = ProjectSearchEngine(
        provider="huggingface",
        csv_file=config.INPUT_CSV
    )

    # CSV laden
    print("📊 Lade CSV-Daten...")
    engine.load_and_clean()

    # Index-Info anzeigen
    info = engine.get_index_info()
    print("")
    print("📈 System-Informationen:")
    print(f"   • CSV-Zeilen: {info['csv_rows']:,}")
    print(f"   • Indizierte Vektoren: {info['total_vectors']:,}")
    print(f"   • Embedding-Dimension: {info['dimension']}")

    if info['csv_rows'] > 0 and info['total_vectors'] > 0:
        coverage = (info['total_vectors'] / info['csv_rows']) * 100
        print(f"   • Index-Abdeckung: {coverage:.1f}%")

    print("")
    print("🌐 Starte Gradio-Oberfläche...")
    print("")
    print("👉 Klicken Sie auf den generierten Link unten!")
    print("   (URL beginnt mit https://...gradio.live)")
    print("")

    # Gradio UI starten
    try:
        from src.app_with_logging import build_ui_with_logging
        print("   🔍 Verwende Debug-Version mit Live-Logging")
        demo = build_ui_with_logging(engine)
    except ImportError:
        print("   ℹ️ Verwende Standard-UI")
        demo = build_ui(engine)

    demo.launch(
        share=True,      # Öffentlicher Link (Colab-kompatibel)
        debug=False,
        show_error=True,
        server_name="0.0.0.0",
        server_port=7860
    )

except Exception as e:
    print(f"❌ FEHLER: {e}")
    import traceback
    traceback.print_exc()
    print("")
    print("Mögliche Lösungen:")
    print("   1. Starten Sie die Runtime neu: Runtime → Restart runtime")
    print("   2. Führen Sie alle Zellen erneut aus")
    print("   3. Prüfen Sie ob genug RAM verfügbar ist (>8 GB empfohlen)")
    print("   4. Verwenden Sie eine GPU-Runtime für bessere Performance")

## 💡 Nutzungshinweise

### Suchmodi

- **Hybrid** (empfohlen): Kombiniert semantische und Keyword-Suche
- **Semantic**: Reine KI-basierte Vektorsuche
- **Keyword**: Schnelle textbasierte Suche

### Beispielsuchen

```
Künstliche Intelligenz Hochschule Bayern
Wasserstoff Energie NRW 2020-2025
Quantencomputing Forschung
Klimawandel Digitalisierung
Medizintechnik Berlin
```

### Features

- ✅ **300.000+ Förderprojekte** durchsuchbar
- ✅ **Semantische Suche** findet thematisch ähnliche Projekte
- ✅ **KI-Analyse** generiert Zusammenfassungen
- ✅ **Projekt-Details** per FKZ-Auswahl
- ✅ **Statistiken** zu Fördersummen und Zeiträumen

---

## 🛠️ Fehlerbehebung

### Problem: "Out of Memory"

**Lösung**: Starten Sie die Runtime neu und führen Sie nur die nötigen Zellen aus.

```python
# Runtime neu starten
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)
```

### Problem: "API Key ungültig"

**Lösung**: Überprüfen Sie Ihren GROQ API Key:

```python
import os
print(f"API Key: {os.environ.get('GROQ_API_KEY', 'NICHT GESETZT')}")
```

### Problem: "Index nicht gefunden"

**Lösung**: Führen Sie Schritt 2 (Download) erneut aus.

```python
# Prüfe Index-Dateien
from pathlib import Path
print(f"Index existiert: {Path('data/vector_hf.index').exists()}")
print(f"Mapping existiert: {Path('data/embeddings_map_hf.json').exists()}")
```

### Problem: "HuggingFace Model Download langsam"

**Lösung**: Das erste Laden des Modells dauert 1-2 Minuten. Das ist normal.

---

## ℹ️ Weitere Informationen

### Links

- 📦 [GitHub Repository](https://github.com/dgaida/rag_foerderkatalog)
- 📖 [Dokumentation](https://github.com/dgaida/rag_foerderkatalog#readme)
- 🐛 [Issues](https://github.com/dgaida/rag_foerderkatalog/issues)
- 💬 [Discussions](https://github.com/dgaida/rag_foerderkatalog/discussions)

### Technische Details

- **Python**: 3.11+
- **Embeddings**: HuggingFace (intfloat/e5-small-v2, 384 dim)
- **Vector DB**: FAISS (CPU)
- **LLM**: GROQ API
- **UI**: Gradio 4.0+

### Lizenz

MIT License - siehe [LICENSE](https://github.com/dgaida/rag_foerderkatalog/blob/master/LICENSE)

---

**© 2025 RAG Förderkatalog** | v0.2.0
